# Telco Customer Churn: A Full Data Mining Case Study

**Business problem:** a telecom company is losing customers (churn). We have historical data on ~7,000 customers — who left, who stayed, and everything about their account (services, contract, billing). The goal: understand *why* customers churn, predict *who* will churn next, and surface patterns that could inform a retention strategy.

This notebook works through the **full data mining pipeline** taught in CSE5DMI, on one real dataset:

| Section | CSE5DMI week | Technique |
|---|---|---|
| 1 | Week 2 | Data understanding, attribute types, data quality |
| 2 | Week 2 / 4 | Preprocessing, cleaning, feature engineering |
| 3 | Week 3 / 4 | Decision tree classification, pruning, evaluation |
| 4 | Week 5 | Precision/recall, ROC/AUC, handling class imbalance |
| 5 | Week 6 | Naive Bayes & SVM, multi-model comparison |
| 6 | Week 7 / 8 | Association rule mining (Apriori), interestingness |
| 7 | Week 9-11 | Customer segmentation via clustering |
| 8 | Week 1 / 5 | Business conclusions (SILO 5: communicate to non-technical audiences) |

**Dataset:** IBM Telco Customer Churn (public, no login required to download).
**How to run:** open this notebook in Google Colab (see the repo README for the "Open in Colab" link once it's on GitHub), then Runtime → Run all, filling in the `# TODO` cells as you go.

---
> 💡 **How to use the TODOs:** each TODO cell is deliberately left for you to fill in — that's where the actual learning happens. A hint is given, but not the answer. If you get stuck, this is exactly the kind of thing worth asking a study partner (or me) to walk through step by step.

In [ ]:
# Setup — run this first
!pip install -q mlxtend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_curve, roc_auc_score
)
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42  # keep this fixed everywhere for reproducible results

## Section 1 — Data Understanding & Quality (Week 2)

Before touching any algorithm, we need to know what we're working with: what type is each attribute, and what quality problems exist?

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

In [ ]:
df.info()

### TODO 1.1 — Classify the attribute types

Using the NOIR framework from Week 2, classify at least 6 columns below. Write your answer as a comment — this isn't graded by code, it's a check on your own understanding.

Hint: `tenure` is measured in months with a true zero — what type is that? `Contract` (Month-to-month / One year / Two year) has an order to it — is that ordinal or nominal? `customerID` carries no information at all beyond uniqueness — what does that make it for modelling purposes?

In [ ]:
# TODO 1.1: classify these columns (edit this dict)
attribute_types = {
    "customerID": "???",      # hint: unique identifier, not a real attribute for modelling
    "gender": "???",
    "SeniorCitizen": "???",   # hint: stored as 0/1 — but what does it represent conceptually?
    "tenure": "???",
    "Contract": "???",        # hint: does order matter here?
    "MonthlyCharges": "???",
    "Churn": "???",           # this is our target/class label
}
attribute_types

### TODO 1.2 — Find the data quality problems

Week 2 taught you to look for: noise, outliers, missing values, duplicates, and "wrong" data (a value technically present but not usable, e.g. a number stored as text).

Run the checks below and inspect the output. `TotalCharges` has a well-known problem in this dataset — find it.

In [ ]:
# Duplicates
print("Duplicate rows:", df.duplicated().sum())

# Missing values (the pandas way — but this dataset hides its missing values differently, keep looking)
print("\nNulls per column:\n", df.isnull().sum().sum())

# TODO: TotalCharges LOOKS numeric but pandas read it as an object (string) column.
# Investigate why. Hint: try pd.to_numeric(df['TotalCharges'], errors='coerce')
# and see how many values become NaN that weren't NaN before.

# your investigation here:

## Section 2 — Preprocessing & Feature Engineering (Week 2 / 4)

In [ ]:
# Fix TotalCharges (worked example — this one's done for you)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Rows with missing TotalCharges after conversion:", df['TotalCharges'].isnull().sum())

# These are new customers with tenure = 0 (they haven't been billed yet) — sensible to impute as 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop the ID column — it carries no predictive information (you classified this in TODO 1.1)
df = df.drop(columns=['customerID'])

### TODO 2.1 — Bucket `tenure` into groups

This is the same technique as your Week 4 lab (bucketing `age` into groups on the census dataset). Create a new column `tenure_group` with bins: `0-12`, `13-24`, `25-48`, `49-60`, `61-72` (months).

Hint: use `pd.cut()` with the `bins` and `labels` parameters.

In [ ]:
# TODO 2.1: create df['tenure_group']

### TODO 2.2 — Encode categorical variables

Most columns here are categorical strings (`Yes`/`No`, service types, etc.) — sklearn models need numbers. Use `pd.get_dummies()` (one-hot encoding, same as your IFU/DM lab pattern) for the feature columns, and a simple `LabelEncoder` (or manual `.map()`) for the binary target `Churn`.

Keep a separate copy of the *uncleaned* categorical columns (`df_original`) before encoding — you'll want the readable version for Section 6 (association rules).

In [ ]:
df_original = df.copy()  # keep a readable copy for later

# TODO 2.2: encode Churn as 0/1, one-hot encode the rest of the categorical columns
# y = ...
# X = ...

In [ ]:
# Train/test split (Week 4: holdout method)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.2%}, test: {y_test.mean():.2%}")

## Section 3 — Decision Tree Classification (Week 3 / 4)

### TODO 3.1 — Manual Gini check (theory → code)

Before trusting sklearn, prove to yourself you understand what it's doing. Compute the Gini impurity of the **training set's target column** by hand (i.e. in code, using the formula, not a library function), then compare it to what you'd expect.

Formula: `Gini = 1 - sum(p_i^2)` for each class i.

In [ ]:
# TODO 3.1: compute Gini impurity of y_train manually
# p_churn = ...
# p_no_churn = ...
# gini = ...
# print(f"Manual Gini: {gini:.4f}")

### TODO 3.2 — Train a decision tree and diagnose over/underfitting

Train **two** trees: one with no depth limit (likely to overfit) and one with `max_depth=4` (constrained). For each, print training accuracy and test accuracy. Which shows a bigger train/test gap? That's your overfitting signal from Week 4.

In [ ]:
# TODO 3.2:
# tree_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
# tree_full.fit(X_train, y_train)
# ... print train vs test accuracy for both trees

In [ ]:
# Visualise your best tree (top 3 levels only, full tree is unreadable)
# TODO: replace `tree_full` with whichever tree you decide is your "final" choice from 3.2
plt.figure(figsize=(20, 8))
plot_tree(tree_full, max_depth=3, feature_names=X.columns, class_names=['No Churn', 'Churn'], filled=True, fontsize=8)
plt.show()

## Section 4 — Beyond Accuracy: Precision, Recall, ROC (Week 5)

Churn is imbalanced (~26% churn rate) — Week 5 taught you why accuracy alone is misleading here.

In [ ]:
y_pred = tree_full.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### TODO 4.1 — Plot an ROC curve and compute AUC

Use `roc_curve()` and `roc_auc_score()` from sklearn (they need predicted *probabilities*, not hard labels — use `.predict_proba()`). Plot the curve, and mark the diagonal (random-guess) line for reference, same as the ROC theory from Week 5.

In [ ]:
# TODO 4.1: ROC curve for the decision tree
# y_proba = tree_full.predict_proba(X_test)[:, 1]
# fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# ... plot it, including the diagonal line

## Section 5 — Model Comparison: Naive Bayes & SVM (Week 6)

### TODO 5.1 — Train Naive Bayes and SVM, compare all three on one ROC plot

Train a `GaussianNB()` and an `SVC(probability=True)` on the same `X_train`/`y_train`. Plot all three models' ROC curves on the same axes (decision tree, Naive Bayes, SVM) — this is exactly the "comparing models via ROC/AUC" concept from Week 5-6.

Note: SVM training can be slow on the full feature set — if it's too slow in Colab, scale down with `StandardScaler` first (SVMs are sensitive to feature scale, unlike trees).

In [ ]:
# TODO 5.1: train NB and SVM, plot combined ROC comparison

## Section 6 — Association Rule Mining (Week 7 / 8)

Which combinations of services and contract types are *associated* with churn? This uses the exact Apriori workflow from your Week 7-8 material — support, confidence, and (new this section) **lift** as an interestingness measure.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Build a "basket" of Yes/No service columns + churn, using the readable df_original from earlier
basket_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies', 'Contract', 'Churn']
basket_df = df_original[basket_cols].copy()
basket_df.head()

### TODO 6.1 — Turn this into transaction format and run Apriori

Each row needs to become a *set of items* (e.g. `{'TechSupport_No', 'Contract_Month-to-month', 'Churn_Yes'}`), the same shape as the Bread/Milk/Nappies transaction table from Week 7.

Hint: for each column, create item labels like `f"{column}={value}"`, collect them into a list per row, then use `TransactionEncoder` to one-hot encode into the format `apriori()` expects.

In [ ]:
# TODO 6.1: build transactions and run apriori
# transactions = [...]
# te = TransactionEncoder()
# te_array = te.fit(transactions).transform(transactions)
# basket_encoded = pd.DataFrame(te_array, columns=te.columns_)
# frequent_itemsets = apriori(basket_encoded, min_support=0.1, use_colnames=True)

### TODO 6.2 — Generate rules and find the most interesting ones

Run `association_rules()` with `metric="confidence"`, `min_threshold=0.5`. Then **filter for rules where the consequent is `Churn=Yes`** and sort by `lift` descending. By hand (in a markdown cell below your code), explain what your top rule means in plain English, and whether support, confidence, or lift is doing the most work to make it "interesting" — this connects directly to the Week 8 practice question on interestingness measures.

In [ ]:
# TODO 6.2: generate and filter rules

## Section 7 — Customer Segmentation via Clustering

> 📌 **Come back to this section once your lectures cover clustering (K-means, hierarchical) — the syllabus places this a few weeks ahead of where you likely are now.** The scaffold is here so you don't lose the thread of the project.

### TODO 7.1 — K-means on tenure, MonthlyCharges, TotalCharges

Standardise these three columns first (K-means is distance-based, so scale matters — same reasoning as SVM in Section 5). Use the **elbow method** (plot inertia vs k for k=2..10) to choose a sensible number of clusters, then fit K-means with that k and profile each resulting cluster (mean tenure/spend/churn rate per cluster) — give each segment a business-friendly name (e.g. "new & price-sensitive", "loyal high-value").

In [ ]:
# TODO 7.1: K-means segmentation

## Section 8 — Conclusions (Week 1 / SILO 5)

### TODO 8.1 — Write this up for a non-technical stakeholder

In 150-300 words below, summarise: (1) which customers are most likely to churn and why, (2) your model's real-world usefulness (is 80% accuracy actually good here, given the class imbalance?), (3) one concrete retention action the business could take based on your association rules, (4) one limitation of this analysis you'd flag before anyone acts on it.

This is literally what SILO 5 of your subject asks you to be able to do — write it as if for your course coordinator, not a data scientist.

_Your conclusion here._